# Exercise A — Compare Three Chunking Strategies
Chunking decides retrieval quality more than any other single knob. Here you split the same text three ways and see which gives the cleanest retrieval.

**Real version:** run this on the actual `ERP-2008-chapter4.pdf` text. Here we use a paragraph of it so the scaffold runs offline.

In [ ]:
# Offline mock so this scaffold runs with NO API key / NO network.
# For real practice, replace `embed()` with your real embedder (InHouseEmbeddings,
# SentenceTransformer, etc.) and `llm()` with a real model call.
import numpy as np, re
_STOP=set("the a an to of and or is are be for in on at by with from as that this it its".split())
def _tok(t): return [w for w in re.findall(r"[a-z0-9]+",t.lower()) if w not in _STOP and len(w)>2]
def embed(texts):
    if isinstance(texts,str): texts=[texts]
    out=[]
    for t in texts:
        v=np.zeros(256)
        for w in _tok(t): v[abs(hash(w))%256]+=1
        n=np.linalg.norm(v); out.append(v/n if n else v)
    return np.array(out)
def cos(a,b): return float(a@b)

# A small corpus standing in for chunks of ERP-2008-chapter4.pdf (health-care economics).
CORPUS = [
 ("Demand for health care is derived from the value of improved health, not the procedures themselves.","demand"),
 ("Health can be defined by longevity (length of life) and quality of life.","demand"),
 ("National health spending reached over 7000 dollars per capita and about 16 percent of GDP.","spending"),
 ("Medical technology accounts for about half of long-term health spending growth.","spending"),
 ("Medicare, enacted in 1965, covers people aged 65 and older; Part D is the drug benefit.","medicare"),
 ("Medicaid, established in 1965, is a program for low-income individuals, administered by states.","medicaid"),
 ("Moral hazard is the tendency to overuse care when insurance covers most of the cost.","moral_hazard"),
 ("Adverse selection is when insurance is most attractive to those most likely to need it.","insurance"),
 ("Health Savings Accounts use pre-tax dollars with high-deductible plans to reduce routine-care reliance.","hsa"),
 ("The proposed standard deduction for health insurance would be a flat 15000 dollars per family.","tax"),
]
texts=[c[0] for c in CORPUS]; sections=[c[1] for c in CORPUS]
print("Mock corpus ready:", len(texts), "chunks.")

In [ ]:
PARAGRAPH = (
 "Demand for health care is unlike demand for most products. The desire for health care "
 "is not derived directly from consuming medical procedures; it comes from the value of "
 "improved health. Health can be defined along two dimensions: longevity, the length of "
 "life, and the quality of life. A person derives value from quality of life directly "
 "because health affects the enjoyment of goods and leisure, and indirectly because "
 "health enhances productivity, which is rewarded in the labor market through higher wages."
)

def chunk_fixed(text, size, overlap=0):
    words = text.split(); chunks=[]; start=0
    while start < len(words):
        chunks.append(" ".join(words[start:start+size]))
        start += max(1, size-overlap)
    return [c for c in chunks if c]

def chunk_sentences(text):
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]

strategies = {
    "fixed_20":        chunk_fixed(PARAGRAPH, 20, 0),
    "fixed_20_ov5":    chunk_fixed(PARAGRAPH, 20, 5),
    "sentences":       chunk_sentences(PARAGRAPH),
}
for name, chunks in strategies.items():
    print(f"{name}: {len(chunks)} chunks")

In [ ]:
# Retrieve the best chunk for a query under each strategy
QUERY = "what are the two dimensions of health?"
qv = embed(QUERY)[0]
for name, chunks in strategies.items():
    cv = embed(chunks)
    best = int(np.argmax([cos(qv, v) for v in cv]))
    print(f"\n[{name}] best chunk:")
    print("  ", chunks[best])

### Observe & decide
- Which strategy's best chunk most cleanly contains 'longevity and quality of life'?
- Sentence chunks are precise but may lack context; fixed chunks carry more context but can bury the answer; overlap rescues answers split across a boundary.
**Your turn:** run this on the full PDF text and score with several queries. Tie back to Specialist Track Phase 2.